### Import Libraries & API Keys

In [4]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if OPENAI_API_KEY is None:
    raise Exception("API Key is missing")

### Set up Pushover

In [ ]:
#step 2a -> Set up account in your browser
#step 2b -> Set up the app on your iPhone / Android, log into the same account
#step 2c -> In the browser createw an "Application/API Token"
#Step 2d -> Copy your User Key and API Token into the .env file,
#like this but with your own keys:
#PUSHOVER_USER=xxxxxxx
#PUSHOVER_TOKEN=yyyyyy

#Save changes to the .env file
#Run the test to manually send a notification from Pushover in your browser to your phone

In [6]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

In [8]:
#use this (!in private!) to test that your keys have been loaded

# print(pushover_user)
# print(pushover_token)

In [12]:
#Test Pushover
import requests

def send_notification(message: str):
    payload = {
        "user": pushover_user,
        "token": pushover_token,
        "message": message
    }
    requests.post(pushover_url, data=payload)

In [ ]:
send_notification("Hello to myself from my AI engineering training")

### Step 3: Describe Pushover as an LLM tool

In [13]:
send_notification_function = {
    "name": "send_notification",
    "description": "Sends a push notification to the user's phone via Pushover. Use this to alert the user about important events, completed tasks, or time-sensitive information",
    "parameters" : {
        "type": "object",
        "properties": {
            "message": {
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

### Step 4: Add Pushover to the list of tools for the LLM

In [15]:
tools = [{"type": "function", "function": send_notification_function}]

### Step 5: Calling the tool from an LLM

In [16]:
client = OpenAI()
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Please send me a notification on what amazing progress\
         I'm making on teh AI Engineering training by SuperDataScience."}],
    tools=tools
)

#check if model wants to call a tool
message = response.choices[0].message

In [20]:
# print(message)

In [ ]:
if message.tool_calls:
    tool_call = message.tool_calls[0]
    import json
    args = json.loads(tool_call.function.arguments) 
    send_notification(args["message"])
else:
    print(message.content)   